# Phase 3: testing PaDiM on the real model

**CRISP-DM stage:** Data preparation

**Why this exists:** PaDiM's inverse step crashes on the Kaggle GPU when many tables are inverted at once. On made-up tables, the processor did the job quickly and accurately. This tests that route on the real model, at image size 256, before relying on it.

**Runs on:** Kaggle with the GPU and internet on, to install anomalib and fetch the pretrained networks. On the laptop it only checks the code, with untrained networks at a tiny size.

**Which photos:** good Vial training photos only. No test photos are read, so half A and half B are not involved. Scores on training photos only prove the model runs. They say nothing about how well it finds defects.

In [ ]:
!pip install anomalib==2.7.0

## 1. Settings

- **Image size 256:** the main size chosen for the experiments.
- **ResNet-18 and Wide ResNet-50:** the two networks to test. The third, EfficientNet, has its own short test later. Each uses anomalib's default number of kept features: 100 and 550.
- **`layers=["layer1", "layer2", "layer3"]`:** anomalib's default for PaDiM.
- **8 photos, then all 291:** 8 is a quick first check, and 291 is the largest real job, because memory problems only show at full size.
- **Seed 42:** PaDiM keeps a random choice of features, so the seed makes this repeatable.
- **8 photos per batch:** the same as the size check.

In [ ]:
import csv
import gc
import os
import sys
import time
from pathlib import Path

import torch
from anomalib.data.utils import read_image
from anomalib.models import Padim
from anomalib.models.image.padim.torch_model import PadimModel

USING_GPU = torch.cuda.is_available()

if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    IMAGE_SIZE = 256
    BATCH_SIZE = 8
    PHOTO_COUNTS = [8, 291]
    PRETRAINED = True
else:
    # The laptop only proves the code runs: tiny photos and untrained networks
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    IMAGE_SIZE = 64
    BATCH_SIZE = 2
    PHOTO_COUNTS = [2, 5]
    PRETRAINED = False

NETWORKS = ["resnet18", "wide_resnet50_2"]
LAYERS = ["layer1", "layer2", "layer3"]
SEED = 42
# anomalib adds these two values along the diagonal of each table before inverting
DIAGONAL_ADDITION = 0.01 + 0.00001
POSITIONS_CHECKED = 4
RESULTS_FILE = "padim_real_results.csv"

vial_folder = None
for train_folder in INPUT_FOLDER.rglob("train"):
    if train_folder.parent.name == "vial":
        vial_folder = train_folder.parent
if vial_folder is None:
    raise FileNotFoundError("Could not find the Vial folder")

all_paths = sorted((vial_folder / "train" / "good").glob("*.png"))
print("Vial training photos found:", len(all_paths))

with open(RESULTS_FILE, "w", newline="") as results_file:
    csv.writer(results_file).writerow(["network", "photos", "measure", "value"])

# Total processor memory, on Kaggle's Linux system only
if sys.platform == "linux":
    total_memory_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    print("Processor memory in GB:", round(total_memory_gb, 1))

## 2. Helper functions

`processor_peak_gb` reads the most processor memory used so far in this session. It never goes down, so a later test's number can include an earlier test's peak. It only works on Kaggle's Linux system.

`largest_inverse_error` rebuilds the table at a few positions, using anomalib's own `_cov` calculation and its two diagonal additions, then checks that the table times its inverse gives the identity table.

In [ ]:
def record(network, photos, measure, value):
    """Save one measured number straight away, and print it."""
    with open(RESULTS_FILE, "a", newline="") as results_file:
        csv.writer(results_file).writerow([network, photos, measure, value])
    print(network, photos, measure, value, flush=True)


def prepare_batch(paths):
    """Read photos and apply anomalib's default recipe at the chosen size."""
    recipe = Padim.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform
    photos = []
    for path in paths:
        photos.append(read_image(path, as_tensor=True))
    return recipe(torch.stack(photos)).to(DEVICE)


def wait_for_gpu():
    """Wait for the GPU to finish, so the timer does not stop early."""
    if USING_GPU:
        torch.cuda.synchronize()


def clear_memory():
    """Free memory, so one test cannot affect the next one's peak."""
    gc.collect()
    if USING_GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def gpu_peak_gb():
    """Return the most GPU memory used since the last reset."""
    if USING_GPU:
        return round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return "no gpu"


def processor_peak_gb():
    """Return the most processor memory used so far in this session."""
    if sys.platform == "linux":
        import resource
        # On Linux this number is in kilobytes
        return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)
    return "not measured"


def largest_inverse_error(gaussian, embeddings):
    """Check table times inverse at a few positions, on the processor."""
    photos, features, height, width = embeddings.shape
    flat = embeddings.view(photos, features, height * width)
    identity = torch.eye(features)
    step = (height * width) // POSITIONS_CHECKED
    largest = 0.0
    for position in range(0, height * width, step):
        table = gaussian._cov(flat[:, :, position], rowvar=False) + DIAGONAL_ADDITION * identity
        product = table @ gaussian.inv_covariance[position]
        error = float((product - identity).abs().max())
        if error > largest:
            largest = error
    return largest

## 3. The test

Each test follows the route being checked:

1. **Features on the GPU.** `PadimModel` in training mode stores each batch's features in `memory_bank`.
2. **Statistics on the processor.** The features move to the processor, and anomalib's own `gaussian.fit` works out the average and variation, then the inverse. Its code is not changed. It simply runs on the processor, because that is where the numbers now are.
3. **Scoring on the GPU.** The answers move back to the GPU, and 8 training photos are scored, to prove it runs and gives finite numbers.

**Alternative not used:** doing only the inverse on the processor would need a copy of anomalib's maths, with more code to check. This route moves the whole statistics step, which is slightly wider, but it is the same calculation.

**Stopping:** a GPU error stops the whole run, because it jams the GPU. Running out of GPU memory is recorded, then the run moves on. ResNet-18 runs first, so Wide ResNet-50 cannot affect it.

In [ ]:
def run_test(network, photo_count):
    """Run the whole route once for one network and number of photos."""
    torch.manual_seed(SEED)
    model = PadimModel(backbone=network, layers=LAYERS, pre_trained=PRETRAINED).to(DEVICE)
    model.train()
    # The pretrained network must stay unchanged while features are collected
    model.feature_extractor.eval()

    # Evenly spread photos show more variety than the first few
    step = max(1, len(all_paths) // photo_count)
    paths = all_paths[::step][:photo_count]
    record(network, photo_count, "photos_used", len(paths))

    started = time.perf_counter()
    for start in range(0, len(paths), BATCH_SIZE):
        model(prepare_batch(paths[start:start + BATCH_SIZE]))
    wait_for_gpu()
    record(network, photo_count, "seconds_features_including_reading", round(time.perf_counter() - started, 2))
    record(network, photo_count, "peak_gpu_gb_features", gpu_peak_gb())

    # Moves the features to the processor, and frees them from the GPU
    embeddings = torch.vstack(model.memory_bank).cpu()
    model.memory_bank = []
    clear_memory()

    started = time.perf_counter()
    model.gaussian.fit(embeddings)
    record(network, photo_count, "seconds_fit_on_processor", round(time.perf_counter() - started, 2))
    record(network, photo_count, "processor_peak_gb_so_far", processor_peak_gb())
    record(network, photo_count, "largest_inverse_error", round(largest_inverse_error(model.gaussian, embeddings), 6))
    del embeddings

    # The answers are needed on the GPU for scoring
    model.gaussian.to(DEVICE)
    clear_memory()
    model.eval()
    batch = prepare_batch(all_paths[:BATCH_SIZE])
    model(batch)
    wait_for_gpu()
    started = time.perf_counter()
    scored = model(batch)
    wait_for_gpu()
    record(network, photo_count, "seconds_per_photo_scoring", round((time.perf_counter() - started) / len(batch), 4))
    record(network, photo_count, "peak_gpu_gb_scoring", gpu_peak_gb())
    record(network, photo_count, "scores_all_finite", bool(torch.isfinite(scored.anomaly_map).all()))


stop_everything = False
for network in NETWORKS:
    for photo_count in PHOTO_COUNTS:
        clear_memory()
        try:
            run_test(network, photo_count)
        except torch.cuda.OutOfMemoryError:
            record(network, photo_count, "outcome", "out of memory")
            continue
        except torch.AcceleratorError as gpu_error:
            record(network, photo_count, "outcome", "gpu error: " + str(gpu_error).split(chr(10))[0])
            stop_everything = True
            break
        record(network, photo_count, "outcome", "worked")
    if stop_everything:
        break
print("Finished")